# 🚀 Ekegusii (Kisii) Parallel Translation Pipeline

This notebook runs the dynamic Retrieval-Augmented Ekegusii translation pipeline on Google Colab's GPU/CPU runtime.

### Step 0: Clone Repository
Clone the repository to fetch the required generator scripts and templates, then change into the project directory. If the directory already exists, it will pull the latest updates.

In [ ]:
import os
%cd /content
if not os.path.exists('public-service-anouncement-MT'):
    !git clone https://github.com/SamAbr/public-service-anouncement-MT.git
    %cd public-service-anouncement-MT
else:
    print('Repository directory already exists. Resetting and pulling latest updates...')
    %cd public-service-anouncement-MT
    !git reset --hard HEAD
    !git pull

### Step 1: Install Dependencies
Install the required translation, tracking, and language ID libraries.

In [ ]:
!pip install transformers sentencepiece tqdm pandas torch fasttext nltk gspread gspread-dataframe sentence-transformers

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive to persist checkpoints...")
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/psa_generator"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "psa_parallel_dataset.csv")
print(f"Output dataset path set to: {OUTPUT_CSV}")

### Step 2: Dynamic Retrieval-Augmented Ekegusii Translation
Run the translation below. This uses a custom **Retrieval-Augmented Few-Shot Prompting** architecture that dynamically retrieves the top 3 most semantically similar verified sentence pairs from a local seed corpus to feed as context into the LLM, ensuring premium translation quality.

In [ ]:
# Run Ekegusii translation in phases to avoid timeout. Specify model (e.g. gpt-5-mini)
EKEGUSII_API_KEY = "YOUR_OPENAI_OR_AZURE_KEY"

# Number of records to translate in this phase (e.g. 5000, 10000, or set to None to translate all)
PHASE_LIMIT = 5000

if EKEGUSII_API_KEY and EKEGUSII_API_KEY != "YOUR_OPENAI_OR_AZURE_KEY":
    print(f"Running Ekegusii translation (Model: gpt-5-mini, Phase Limit: {PHASE_LIMIT})...")
    limit_flag = f"--limit {PHASE_LIMIT}" if PHASE_LIMIT is not None else ""
    !python translate_ekegusii_llm.py --input output/psa_parallel_dataset.csv --output "{OUTPUT_CSV}" --api-key {EKEGUSII_API_KEY} --model gpt-5-mini --workers 5 --batch-size 50 {limit_flag}
else:
    print("Please provide your API key to run Ekegusii translation.")


### Step 3: Download Parallel Dataset
Download the final parallel CSV file to your local computer.

In [ ]:
import shutil
shutil.copy(OUTPUT_CSV, 'output/psa_parallel_dataset.csv')
from google.colab import files
files.download('output/psa_parallel_dataset.csv')

### Step 4: Save, Commit, and Push to GitHub
Upload the generated CSV file directly to your GitHub repository. 

**Note:** You will need to enter your GitHub Personal Access Token (PAT) securely when prompted.

In [ ]:
# Ensure latest checkpoint from Drive is copied to workspace before Git operations
import shutil
import os

shutil.copy(OUTPUT_CSV, 'output/psa_parallel_dataset.csv')

# Ensure we are in the repository directory
%cd /content/public-service-anouncement-MT

# Configure Git credentials and Target Repository / Fork
FORK_OWNER = "SamAbr"  # Replace with your friend's GitHub username if different
GIT_NAME = "SamAbr"
GIT_EMAIL = "samuelab85042018@gmail.com"
GIT_TOKEN = "YOUR_GITHUB_TOKEN"

!git config --global user.name "{GIT_NAME}"
!git config --global user.email "{GIT_EMAIL}"

# Configure remote URL to push to target repository/fork
!git remote set-url origin https://{GIT_TOKEN}@github.com/{FORK_OWNER}/public-service-anouncement-MT.git

# Stage, commit, and push the dataset
!git add output/psa_parallel_dataset.csv
!git commit -m "Upload generated parallel datasets from Colab GPU run"
!git push origin HEAD:main --force
